In [1]:
!pip install datasets
!pip uninstall -y protobuf grpcio grpcio-status google-colabqlviz

Found existing installation: protobuf 3.20.3
Uninstalling protobuf-3.20.3:
  Successfully uninstalled protobuf-3.20.3
Found existing installation: grpcio 1.53.1
Uninstalling grpcio-1.53.1:
  Successfully uninstalled grpcio-1.53.1
Found existing installation: grpcio-status 1.48.2
Uninstalling grpcio-status-1.48.2:
  Successfully uninstalled grpcio-status-1.48.2


In [2]:
!pip install protobuf==3.20.3 grpcio==1.53.1 grpcio-status==1.48.2

  Using cached protobuf-3.20.3-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl.metadata (679 bytes)
  Using cached grpcio-1.53.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.8 kB)
  Using cached grpcio_status-1.48.2-py3-none-any.whl.metadata (1.2 kB)
Using cached protobuf-3.20.3-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl (1.1 MB)
Using cached grpcio-1.53.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (5.0 MB)
Using cached grpcio_status-1.48.2-py3-none-any.whl (14 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colabsqlviz 0.1.3 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 3.20.3 which is incompatible.


In [80]:

from google.cloud import storage
from google.cloud import bigquery
import os
import re
from tqdm import tqdm
import numpy as np
from collections import defaultdict

import sys
import pandas as pd
import gcsfs
import torch
from transformers import AutoTokenizer, AutoModel
from datasets import Dataset
from itertools import islice

# Initialize BigQuery client
# Set up GCS client
client = storage.Client()
bucket = client.bucket('cdow')

# Define local and GCS paths
gcs_folder = 'leads_env'
local_folder = '/content/leads_env'

# Create local folder


In [4]:

os.makedirs(local_folder, exist_ok=True)

# Download all files from GCS to local folder
blobs = bucket.list_blobs(prefix=gcs_folder)
for blob in blobs:
    rel_path = blob.name[len(gcs_folder)+1:]  # remove folder prefix
    if rel_path:  # skip folder itself
        local_path = os.path.join(local_folder, rel_path)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        blob.download_to_filename(local_path)


In [4]:


sys.path.insert(0, '/content/leads_env')


In [5]:


client = bigquery.Client()

project_id = "expanded-nebula-754"
dataset_id = "sandbox_crdow"
table_id = "leads_dataset"
table_ref = f"{project_id}.{dataset_id}.{table_id}"
destination_uri = "gs://cdow/leads_data/leads_scoring.csv"

In [19]:


# Configure extract job
extract_job = client.extract_table(
    table_ref,
    destination_uri,
    location="US", # Change if your dataset is in a different location
    job_config=bigquery.job.ExtractJobConfig(
        destination_format="CSV",
        print_header=True,
        field_delimiter=","
    )
)

# Wait for job to complete
extract_job.result()
print(f"Exported {table_ref} to {destination_uri}")


Exported expanded-nebula-754.sandbox_crdow.leads_dataset to gs://cdow/leads_data/leads_scoring.csv


In [52]:



GCS_PATH = '/content/leads_env'
# Set Hugging Face cache to GCS folder
HF_CACHE = f"{GCS_PATH}/hf_cache"
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE


model_name = "xlm-roberta-base"


In [49]:


# Initialize GCS filesystem
fs = gcsfs.GCSFileSystem()

# Path to your CSV in GCS
csv_path = 'cdow/leads_data/leads_scoring.csv'

# Read CSV directly into a pandas DataFrame
with fs.open(f'gs://{csv_path}') as f:
    df = pd.read_csv(f)




In [50]:
for col in df.columns:
    dtype = df[col].dtype
    if pd.api.types.is_string_dtype(dtype) or dtype == object:
        df[col] = df[col].fillna('')
    if pd.api.types.is_integer_dtype(dtype):
      df[col] = df[col].fillna(0)
    if pd.api.types.is_float_dtype(dtype):
      df[col] = df[col].fillna(0.0)

df = df.replace('',"__MISSING__")
columns = df.columns
print(columns)


Index(['opportunities', 'Email', 'Type', 'campaign_name', 'search_terms',
       'source_campaigns', 'departments',
       'sfdc_product_of_greatest_interest_pogis', 'lead_source_most_recents',
       'av_revenue', 'page_views', 'revenue', 'url', 'eventdate', 'is_won',
       'is_closed', 'stage'],
      dtype='object')


In [75]:
def convert_to_categorical(df, col, min_freq=10):
    value_counts = df[col].value_counts()
    frequent_values = value_counts[value_counts >= min_freq].index

    df.loc[:, col] = df[col].apply(lambda x: x if x in frequent_values else "__RARE__")
    df.loc[:, col] = df[col].fillna("__MISSING__")


#df = df.copy()
#df[col] = df[col].apply(lambda x: x if x in frequent_values else "__RARE__")
#df[col] = df[col].fillna("__MISSING__")


    #df[col] = df[col].apply(lambda x: x if x in frequent_values else "__RARE__")
    #df[col] = df[col].fillna("__MISSING__")
    return df




def preprocess_url(url):
    if pd.isna(url) or url.strip() == "":
        return "__MISSING__"

    # Remove protocol
    url = re.sub(r'^https?:\/\/', '', url)

    # Replace common delimiters with spaces
    url = re.sub(r'[\/\.\?\=\&\%\:\_\-]', ' ', url)

    # Remove long numeric strings (likely IDs)
    url = re.sub(r'\b\d{4,}\b', ' ', url)

    # Remove file extensions
    url = re.sub(r'\b(html|php|aspx|jsp|json|xml|txt)\b', ' ', url)

    # Collapse multiple spaces
    url = re.sub(r'\s+', ' ', url).strip()
    #print(url)
    return url




def preprocess_dataframe(df,text_colums):
    # Fill missing
    for col in text_colums:#["email", "location", "campaign", "url"]:
        df.loc[:,col] = df[col].fillna("__MISSING__")

    # Convert sparse fields to categorical if appropriate
    #for col in text_colums:#["email", "location", "campaign"]:
    #    df = convert_to_categorical(df, col, min_freq=10)

    # Preprocess URL
    df.loc[:,"url"] = df["url"].apply(preprocess_url)

    return df



def embed_texts(df,text_colums, pad_len=512):
    all_embeddings = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        # Concatenate all fields into one string
        text = " ".join([str(row[col]) for col in text_colums])#["email", "location", "campaign", "url"]

        # Tokenize
        encoded = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=pad_len,
            return_tensors="pt"
        )

        with torch.no_grad():
            output = model(**encoded)
            # Use CLS token embedding (first token)
            embedding = output.last_hidden_state[:, 0, :].squeeze().numpy()
            all_embeddings.append(embedding)

    return np.array(all_embeddings)


def embed_texts_batched(df_text, pad_len=512, batch_size=64):
   embeddings = {}

   # Preprocess: fill missing values
   #df_text = df_text.fillna("__empty__")
   for batch_start in tqdm(range(0, len(df_text), batch_size)):
       batch_end = min(batch_start + batch_size, len(df_text))
       batch = df_text.iloc[batch_start:batch_end]

       #batch_embeddings = []
       lead_id =''
       for i, row in batch.iterrows():
           i = int(i)
           # Concatenate all columns into one string
           lead_id =  row[df_text.columns[0]]
           sample_key = f'sample{i}'
           text = " ".join([str(row[col]) for col in df_text.columns[1:]])
           # Tokenize and chunk
           tokens = tokenizer.tokenize(text)
           chunks = [tokens[i:i+pad_len] for i in range(0, len(tokens), pad_len)]
           chunk_embeddings = []
           for chunk in chunks:
               encoded = tokenizer.encode_plus(
                   chunk,
                   is_split_into_words=True,
                   return_tensors="pt",
                   padding="max_length",
                   truncation=True,
                   max_length=pad_len
               )
               encoded = {k: v.to(device) for k, v in encoded.items()}
               with torch.no_grad():
                   output = model(**encoded)
                   cls_embedding = output.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
                   chunk_embeddings.append(cls_embedding)
           # Average all chunk embeddings for this row
           row_embedding = np.mean(chunk_embeddings, axis=0)
           #batch_embeddings.append(row_embedding)
           embeddings[sample_key] = {lead_id:row_embedding}#row_embedding
       #embeddings.extend(batch_embeddings)
   return embeddings
   #return np.array(embeddings)



In [48]:

text_colums = [col for col in df.columns if df[col].dtype.name == 'object' or df[col].dtype.name == 'string']
#print(text_colums)
df_text = df[text_colums]
df_text = preprocess_dataframe(df_text,text_colums)
empty_counts = (df == '').sum()
na_counts = df.isna().sum()
dataset = Dataset.from_pandas(df_text)

['opportunities', 'Email', 'Type', 'campaign_name', 'search_terms', 'source_campaigns', 'departments', 'sfdc_product_of_greatest_interest_pogis', 'lead_source_most_recents', 'url', 'eventdate', 'stage']
opportunities                              0
Email                                      0
Type                                       0
campaign_name                              0
search_terms                               0
source_campaigns                           0
departments                                0
sfdc_product_of_greatest_interest_pogis    0
lead_source_most_recents                   0
av_revenue                                 0
page_views                                 0
revenue                                    0
url                                        0
eventdate                                  0
is_won                                     0
is_closed                                  0
stage                                      0
dtype: int64
opportunities      

In [8]:
import google.protobuf
print(google.protobuf.__version__)
#def tokenize_function(examples):
#    combined_text = []
#    for i in range(len(examples[text_colums[0]])):
#      row_text = ' '.join(str(examples[col][i]) for col in text_colums)
#      combined_text.append(row_text)

    #combined_text = [' '.join(str(examples[col][i]) for col in text_colums) for i in range(len(examples(text_colums[0])))]
    #return tokenizer(combined)
#    return tokenizer(combined_text, padding="max_length", truncation=True)


3.20.3


In [76]:
tokenizer = AutoTokenizer.from_pretrained(model_name,cache_dir=HF_CACHE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModel.from_pretrained(model_name,cache_dir=HF_CACHE)
model.to(device)
model.eval()

XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine

In [13]:
def embed_column(column_name,batch_size=128):
  tokenized_dataset = dataset.map(
      lambda examples:tokenizer(examples[column_name],padding="max_length",truncation=True),
      batched=True,
      batch_size=batch_size)

  embeddings = []
  for i in tqdm(range(0, len(tokenized_dataset), batch_size)):
    batch = tokenized_dataset[i:i+batch_size]
    inputs_ids = torch.tensor(batch['input_ids']).to(device)
    attention_mask = torch.tensor(batch['attention_mask']).to(device)
    with torch.no_grad():
      output = model(inputs_ids,attention_mask=attention_mask)
      pooled = output.last_hidden_state.mean(dim=1)
      embeddings.append(pooled.cpu())

  return torch.cat(embeddings)


In [74]:
df_new = df.iloc[100:131]
#print(df_new[df_new.columns[0]])

col = df.iloc[:, 0]
counts = column_data.value_counts()
duplicates = counts[counts > 4]
#print(duplicates)
#print(column_data)
#
results = {}
for val in duplicates.index:
    indices = df.index[col == val].tolist()
    results[val] = (len(indices),indices)

for val, (count, idx_list) in results.items():
  print(f"'{val}'  appears {count}   times at index  {idx_list}")

Streaming output truncated to the last 5000 lines.
'006Hn00001PrxaxIAB'  appears 3   times at index  [120689, 120690, 120691]
'006dk000001h7s5AAA'  appears 3   times at index  [107943, 107949, 107950]
'0061E00001KEnsqQAD'  appears 3   times at index  [161612, 161613, 161614]
'0061E00001KVNRGQA5'  appears 3   times at index  [191162, 191165, 191166]
'006Hn00001LmkFDIAZ'  appears 3   times at index  [107942, 107944, 107945]
'0061E00001LUNJIQA5'  appears 3   times at index  [240960, 240961, 240962]
'006dk0000024hRrAAI'  appears 3   times at index  [174117, 174118, 174119]
'006dk000006aa1VAAQ'  appears 3   times at index  [127566, 127567, 127568]
'006Hn00001Qd0Y2IAJ'  appears 3   times at index  [191163, 191164, 191167]
'006dk000002XQ2bAAG'  appears 3   times at index  [108747, 108754, 108759]
'006Hn00001LnrvrIAB'  appears 3   times at index  [71559, 71560, 71561]
'006Hn00001MRRwLIAX'  appears 3   times at index  [71563, 71564, 71565]
'006dk000000vtMMAAY'  appears 3   times at index  [1204

In [78]:
embeddings = embed_texts_batched(df_text=df_text)
#column_embeddings = {}
#for col in text_colums:
#  column_embeddings[col] = embed_column(col)
#import torch

#embeddings_leads = embed_texts(df, pad_len=512)

100%|██████████| 15081/15081 [2:05:31<00:00,  2.00it/s]


In [92]:
lead_ids = [list(sample.keys())[0]
           for sample in islice(embeddings.values(),100)
           ]
print(lead_ids[:10])

for sample_id, sample_embeddings in islice(embeddings.items(),10):
    lead_id, embedding = next(iter(sample_embeddings.items()))
    print(f"Sample ID: {sample_id}, Lead ID: {lead_id},   Embedding vals:  {embedding[400:410]}")

['006dk000004DBWUAA4', '0061E00001LJPdlQAH', '0061E00001LUGvrQAH', '006Hn00001MEHKQIA5', '006dk000004kIndAAE', '0061E00001LeXgVQAV', '0061E00001LJWr6QAH', '006Hn00001MEFsuIAH', '006Hn00001MEFs4IAH', '0061E00001LLMGfQAP']
Sample ID: sample0, Lead ID: 006dk000004DBWUAA4,   Embedding vals:  [ 0.06878135 -0.08013359 -0.00513154 -0.08173132  0.12016555  0.10821266
  0.10858969  0.01511941 -0.11524993  0.00034199]
Sample ID: sample1, Lead ID: 0061E00001LJPdlQAH,   Embedding vals:  [ 0.0706852  -0.08231337 -0.00774884 -0.08539421  0.1290567   0.11231919
  0.10497285  0.01635542 -0.11353606  0.00161346]
Sample ID: sample2, Lead ID: 0061E00001LUGvrQAH,   Embedding vals:  [ 0.05275506 -0.07476292 -0.01532636 -0.08804269  0.12387564  0.10412152
  0.10890031  0.01458993 -0.11408672  0.0091824 ]
Sample ID: sample3, Lead ID: 006Hn00001MEHKQIA5,   Embedding vals:  [ 0.06721585 -0.07455867 -0.00451089 -0.06872677  0.11548052  0.09907665
  0.09136793  0.02233027 -0.1036944  -0.0009177 ]
Sample ID: samp

In [ ]:
#lead_ids = [list(sample.keys())[0] for sample in embeddings.values()]
#print(lead_ids[:10])
#import pickle
#buffer = io.BytesIO()
#pickle.dump(embeddings, buffer)
#buffer.seek(0)


#client = storage.Client()
#bucket = client.get_bucket('cdow')
#blob = bucket.blob('leads_data/embeddings.pkl')
#blob.upload_from_file(buffer,rewind=True)


In [95]:
#can be saved to .csv
rows = []
for sample_id, sample_embeddings in embeddings.items():
    for lead_id, embedding in sample_embeddings.items():
      row = {
          'sample_id': sample_id,
          'lead_id': lead_id,
          **{f'emb_{i}': val for i, val in enumerate(embedding)}
      }
      rows.append(row)
print(rows[0:5])
df_bucket = pd.DataFrame(rows)


[{'sample_id': 'sample0', 'lead_id': '006dk000004DBWUAA4', 'emb_0': 0.13193749, 'emb_1': 0.11082553, 'emb_2': 0.10691507, 'emb_3': -0.023023862, 'emb_4': 0.0068622665, 'emb_5': -0.057770252, 'emb_6': 0.0076359524, 'emb_7': -0.027884321, 'emb_8': 0.08550921, 'emb_9': -0.12384288, 'emb_10': 0.06747806, 'emb_11': 0.13675682, 'emb_12': 0.003001785, 'emb_13': 0.017195, 'emb_14': 0.036339264, 'emb_15': 0.04368014, 'emb_16': -0.078800835, 'emb_17': -0.045817886, 'emb_18': 0.05053933, 'emb_19': 0.13063759, 'emb_20': -0.028149057, 'emb_21': 0.023979275, 'emb_22': 0.11964572, 'emb_23': 0.10211152, 'emb_24': -0.021133885, 'emb_25': 0.06277994, 'emb_26': -0.116312906, 'emb_27': -0.026838845, 'emb_28': 0.09870709, 'emb_29': 0.06994124, 'emb_30': 0.09903098, 'emb_31': 0.038353875, 'emb_32': 0.11547698, 'emb_33': 0.1533805, 'emb_34': 0.0688037, 'emb_35': 0.018941367, 'emb_36': -0.1626231, 'emb_37': -0.13448362, 'emb_38': 0.0065739877, 'emb_39': 0.096689664, 'emb_40': 0.09583572, 'emb_41': 0.18321428,

In [102]:
#must be saved to parquet
rows = []
for sample_id, sample_embeddings in embeddings.items():
    for lead_id, embedding in sample_embeddings.items():
      rows.append({
          'sample_id': sample_id,
          'lead_id': lead_id,
          'embedding': embedding.astype(np.float32).tolist()
      })


df_bucket = pd.DataFrame(rows)
print(len(df_bucket))

241288


In [104]:

fs = gcsfs.GCSFileSystem()

with fs.open('cdow/leads_data/embeddings.parquet', 'wb') as f:
  df_bucket.to_parquet(f,index=False)


In [105]:
import gc
del df_bucket
del rows
gc.collect()

94

In [106]:
gcs_path = 'gs://cdow/leads_data/embeddings.parquet'
df_bucket = pd.read_parquet(gcs_path,engine='pyarrow')


  sample_id             lead_id  \
0   sample0  006dk000004DBWUAA4   
1   sample1  0061E00001LJPdlQAH   
2   sample2  0061E00001LUGvrQAH   
3   sample3  006Hn00001MEHKQIA5   
4   sample4  006dk000004kIndAAE   

                                           embedding  
0  [0.13193748891353607, 0.11082553118467331, 0.1...  
1  [0.13267530500888824, 0.11007501929998398, 0.1...  
2  [0.1310054361820221, 0.11476866155862808, 0.11...  
3  [0.1304515153169632, 0.10809260606765747, 0.10...  
4  [0.12572309374809265, 0.109453946352005, 0.108...  


In [11]:


texts = {
    "English": "Hello, how are you?",
    "Japanese": "こんにちは、お元気ですか？",
    "Mixed": "Hello こんにちは, how are you お元気ですか？"
}

# Process each string
for label, text in texts.items():
    print(f"\n--- {label} ---")
    inputs = tokenizer(text, return_tensors="pt")
    print("Tokenized input IDs:", inputs["input_ids"])
    print("Attention mask:", inputs["attention_mask"])

    with torch.no_grad():
        outputs = model(**inputs)
    print("\nModel output (last hidden state):")
    print(outputs.last_hidden_state)
    print("Output shape:", outputs.last_hidden_state.shape)




--- English ---
Tokenized input IDs: tensor([[    0, 35378,     4,  3642,   621,   398,    32,     2]])
Attention mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1]])

Model output (last hidden state):
tensor([[[ 0.3657,  0.3019,  0.2148,  ..., -0.2702,  0.2187,  0.0466],
         [ 0.0063,  0.1472, -0.0094,  ...,  0.1803, -0.0184,  0.1766],
         [ 0.0470,  0.0957, -0.0214,  ...,  0.0231,  0.0606,  0.1980],
         ...,
         [ 0.0206,  0.0200,  0.0035,  ...,  0.1479,  0.0385,  0.0515],
         [ 0.0390,  0.0406,  0.0224,  ...,  0.0846,  0.0654,  0.2197],
         [ 0.4569,  0.3744, -0.0292,  ..., -0.8235, -0.0888,  0.2938]]])
Output shape: torch.Size([1, 8, 768])

--- Japanese ---
Tokenized input IDs: tensor([[     0,      6, 192661,     37,   2636, 112861,  60369,     32,      2]])
Attention mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])

Model output (last hidden state):
tensor([[[ 0.2675,  0.1473,  0.1060,  ..., -0.1632,  0.1804,  0.0309],
         [ 0.0036, -0.0015, -0.0236,  ...,  0.2

In [ ]:
#
#each row, should be 1 variable not one variable per column
#
def embed_texts_batched(df_text, pad_len=512, batch_size=128):
   embeddings = []
   # Preprocess: fill missing values
   #df_text = df_text.fillna("__empty__")
   for batch_start in tqdm(range(0, len(df_text), batch_size)):
       batch_end = min(batch_start + batch_size, len(df_text))
       batch = df_text.iloc[batch_start:batch_end]
       batch_embeddings = []
       for _, row in batch.iterrows():
           # Concatenate all columns into one string
           text = " ".join([str(row[col]) for col in df_text.columns])
           # Tokenize and chunk
           tokens = tokenizer.tokenize(text)
           chunks = [tokens[i:i+pad_len] for i in range(0, len(tokens), pad_len)]
           chunk_embeddings = []
           for chunk in chunks:
               encoded = tokenizer.encode_plus(
                   chunk,
                   is_split_into_words=True,
                   return_tensors="pt",
                   padding="max_length",
                   truncation=True,
                   max_length=pad_len
               )
               encoded = {k: v.to(device) for k, v in encoded.items()}
               with torch.no_grad():
                   output = model(**encoded)
                   cls_embedding = output.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
                   chunk_embeddings.append(cls_embedding)
           # Average all chunk embeddings for this row
           row_embedding = np.mean(chunk_embeddings, axis=0)
           batch_embeddings.append(row_embedding)
       embeddings.extend(batch_embeddings)
   return np.array(embeddings)